In [3]:
def metagenePfofileToST(inputFile,outputFile,cdslength=400,expression=50):
    #########################################################
    # prepare gene expression and length
    with open(inputFile,'r') as input:
        gene_infoDict = {}
        for line in input:
            fileds = line.split()
            gene_name = fileds[0].split('|')[0]
            pos = int(fileds[1])
            start = int(fileds[0].split('|')[2])
            end = int(fileds[0].split('|')[3])
            cdsLength = end - start + 1
            # geneLength = int(fileds[0].split('|')[4])
            density = float(fileds[3])
            # filter CDS > 400 nt gene
            if cdsLength > cdslength:
                # key
                key = ':'.join([gene_name,str(cdsLength)])
                gene_infoDict.setdefault(key,0)
                if start <= pos <= end:
                    gene_infoDict[key] += density
                else:
                    pass
            else:
                pass

    # filter CDS expression > 50
    filtedGeneDict = {}
    for key,val in gene_infoDict.items():
        if val > expression:
            meanNorm = val/int(key.split(':')[1])
            filtedGeneDict[key] = [val,meanNorm]
        else:
            pass
    #########################################################
    # Meta-gene analysis from start codon
    mylist = range(-50, 1501)
    rangeDict = dict([i, 0] for i in mylist)
    countDict = dict([i, 0] for i in mylist)

    # open file
    with open(inputFile,'r') as input:
        for line in input:
            fileds = line.split()
            gene_name = fileds[0].split('|')[0]
            pos = int(fileds[1])
            start = int(fileds[0].split('|')[2])
            end = int(fileds[0].split('|')[3])
            cdsLength = end - start + 1
            # geneLength = int(fileds[0].split('|')[4])
            density = float(fileds[3])
            id = ':'.join([gene_name,str(cdsLength)])
            if id in filtedGeneDict:
                reldist = pos - start
                if -50 <= reldist <= 1500:
                    # divide reads at one position by average number of reads per base for this gene
                    reads = density / filtedGeneDict[id][1]
                    rangeDict[reldist] += reads
                    # how often was this position counted in the calculation
                    countDict[reldist] += 1
                else:
                    pass
            else:
                pass
    
    #########################################################
    # output data
    # sort dict
    tupledlist1 = list(rangeDict.items())
    tupledlist1.sort()
    tupledlist2 = list(countDict.items())
    tupledlist2.sort()

    fullDict = {}
    zippedlist = zip(tupledlist1, tupledlist2)
    for elem in zippedlist:
        col0 = elem[0][0]       # list0 col0 = position (K)
        col1 = elem[0][1]       # list0 col1 = norm read number 
        col2 = elem[1][1]       # list1 col1 = how often was position counted

        #normalization2 by frequnecy
        if col2 == 0:
            fullDict[col0] = 0
        else:
            fullDict[col0] = col1 / col2        

    # calculate relative density(position denisty/mean_density)
    new_fullDict = {}
    meanDensity = sum(fullDict.values())/1550
    for key,val in fullDict.items():
        relDensity = val/meanDensity
        new_fullDict[key] = relDensity

    # Finish output
    tupledlist = list(new_fullDict.items())
    tupledlist.sort()

    # output
    outFileP = open(outputFile, 'w')
        
    for elem in tupledlist:
        outFileP.write('\t'.join([str(elem[0]),str(elem[1])]) + '\n')
    outFileP.close()

In [ ]:
import os

# make folder
os.mkdir('./4.metagene-data')

In [4]:
sample = ['ssb1-inter-rep1.map.density.txt','ssb1-inter-rep2.map.density.txt','ssb2-inter-rep1.map.density.txt','ssb2-inter-rep2.map.density.txt',
          'ssb1-trans-rep1.map.density.txt','ssb1-trans-rep2.map.density.txt','ssb2-trans-rep1.map.density.txt','ssb2-trans-rep2.map.density.txt']

# run
for i in range(0,8):
    metagenePfofileToST(''.join(['3.ribo-density-data/',sample[i]]),''.join(['4.metagene-data/',sample[i],'.metegene2StartCodon.txt']),
                        cdslength=400,expression=50)

In [20]:
def slideWindow(inputFile,outputFile,start,end):
    Dict1 = {}
    with open(inputFile, 'r') as input:
        for line in input:
            fields = line.split()
            col0 = int(fields[0])
            col1 = float(fields[1])
            Dict1[col0] = col1

    # calculate window +-20
    outFile = open(outputFile, 'w')

    sum1 = 0
    start = start
    end = end                  #change this value dependent on the genome
    start_sum = start + 20                  #start_sum = 21                        
    end_sum = end - 20                      #end_sum = 4578139
    
    for J in range(start, start_sum + 1):       #lines 1-21
        for X in range(start, J+20+1):        #sum 1-(J+20)
            sum1 += float(Dict1[X])
        # average
        meanDensity = sum1/len(range(start, J+20+1))
        # output
        outFile.write(str(J) + '\t' + str(meanDensity) + '\n')
        sum1 = 0

    for K in range(start_sum + 1, end_sum + 1):         #lines 22-4578139
        for Y in range(K-20, K+20+1):                   #sum (K-20)-(K+20)
            sum1 += float(Dict1[Y])
        # average
        meanDensity = sum1/len(range(K-20, K+20+1))
        # output
        outFile.write(str(K) + '\t' + str(meanDensity) + '\n')
        sum1 = 0

    for L in range(end_sum + 1, end + 1):               #lines 4578140-4578159
        for Z in range(L-20, end + 1):
            sum1 += float(Dict1[Z])
        # average
        meanDensity = sum1/len(range(L-20, end + 1))
        # output
        outFile.write(str(L) + '\t' + str(meanDensity) + '\n')
        sum1 = 0

    outFile.close()

In [21]:
slideWindow(inputFile='test-meta.txt',outputFile='smmoth.txt',start=-50,end=1500)